In [1]:
# These are the libraries you may use for this assignment.
# If you would like to use any other libraries, email james@uwaterloo.ca for permission first.
# Generative AI tools (including coding assistants) are not permitted for this assignment.

from IPython.display import display, Math, Latex

import pandas as pd
import numpy as np
import numpy_financial as npf
import yfinance as yf
import matplotlib.pyplot as plt
import random
from datetime import datetime

## CFM 101 — Group Assignment 2026
### Robo-Advising Challenge: Two Clients, Two Strategies

**Team Number:** XX  
**Team Member Names (three or four):**  Yogi, Olyvia, Lyn, Stepan

Enter the names of all participating members below (one name per line):

Yogi Shah

Olyvia Song

Lyn Khine

Stepan Pokutny

**Submission:** Friday, November 27, 2026, at 9:00 AM (Eastern), via Dropbox.  
**Submit this completed Jupyter notebook only.**

Your team must construct **both** portfolios from the supplied `Tickers.csv` file:

- **Client A — Growth:** $\$1,000,000$ CAD; seek the highest expected return.
- **Client B — Low Risk:** $\$750,000$ CAD; seek the lowest portfolio volatility.

Both clients require 10–20 eligible stocks, at most 15% in any stock and at most 40% in any sector. Read the full assignment handout for all eligibility, fee, currency, and competition rules. Do not hardcode the selected stocks. Check that tickers are valid. The supplied ticker universe will permit feasible portfolios.

**No generative AI may be used in preparing this assignment.**

## Data preparation and shared functions

Read `Tickers.csv`, check ticker eligibility, retrieve the data needed for your strategies, and create any functions you will reuse for both clients. Use closing prices from **November 27, 2026** for the purchase calculations. All portfolio values and weights must be calculated in CAD, with purchase fees paid from the client budget.

In [ ]:
GROUP_NUMBER = "XX"
TICKER_FILE = "Tickers_Example.csv"             # the TAs' file; change to Tickers_Example.csv only for practice

# For practice runs set PURCHASE_DATE to the latest trading day that already has data;
# set it back to "2026-11-27" for the final submission.
PURCHASE_DATE = "2026-11-27"            # buy at this day's close
HIST_START, HIST_END = "2025-11-27", PURCHASE_DATE   # history window for stats and equal-weight comparison
VOLUME_START = "2025-10-01"
VOLUME_END = "2026-10-01"               # yfinance's end date is exclusive, so this covers through Sep 30, 2026

MIN_AVG_VOLUME = 5_000
MIN_TRADING_DAYS = 18                   # months with fewer trading days are ignored in the volume average

FEE_CAP_USD = 2.15
FEE_PER_SHARE_USD = 0.001
TRADING_DAYS = 252 # the total number of trading days in a year

CLIENTS = {
    "A": dict(name="Growth",   budget=1_000_000, n_min=10, n_max=20, max_weight=0.15, max_sector=0.40),
    "B": dict(name="Low Risk", budget=750_000,   n_min=10, n_max=20, max_weight=0.15, max_sector=0.40),
}


# 1. Read Tickers.csv 
def load_tickers(path=TICKER_FILE):
    """Input : path to the ticker csv (no header)
    Output: list[str] of unique, cleaned tickers"""
    tickers = pd.read_csv(path, header=None, names=['Ticker'])
    return tickers['Ticker'].dropna().astype(str).str.strip().str.upper().drop_duplicates().tolist()


# 2. Eligibility 
def filter_ticks(tickers, min_volume=MIN_AVG_VOLUME, min_days_per_month=MIN_TRADING_DAYS):
    """Filter all tickers based on assignment requirements.
    Input : list of tickers
    Output: (list of eligible tickers, DataFrame of info indexed by Ticker with currency, sector, market)"""
    filtered = []
    records = {}

    for i in tickers:
        try:
            ticker = yf.Ticker(i)

            history = ticker.history(start=VOLUME_START, end=VOLUME_END)

            if history.empty:  # ticker is invalid or has no data for the specified date range
                continue

            vol = history['Volume']
            month = history.index.strftime('%Y-%m')  # used to group by month
            # keep only months with at least min_days_per_month trading days
            avg_vol = vol[vol.groupby(month).transform('count') >= min_days_per_month]

            # an empty avg_vol (no month with enough trading days) must be rejected
            if avg_vol.empty or avg_vol.mean() < min_volume:
                continue

            info = ticker.info  # run after the volume check to save runtime
            market = info.get('market')
            # US or Canadian listing, quoted in USD or CAD, company stock (not ETF/fund)
            if ((market == 'us_market' or market == 'ca_market')
                    and (info.get('currency') == 'USD' or info.get('currency') == 'CAD')
                    and (info.get('quoteType') == 'EQUITY')):
                filtered.append(i)
                records[i] = {"currency": info.get('currency'),
                              "sector": info.get('sector') or "Unknown",   # needed for the sector cap
                              "market": market}
        except Exception:
            continue  # skip the invalid tickers or download errors

    return filtered, pd.DataFrame.from_dict(records, orient="index", columns=["currency", "sector", "market"])

# 3. Price and FX data
def load_market_data(tickers, info, start=HIST_START, end=PURCHASE_DATE):
    """Download closing prices (local currency) and the USD->CAD rate.
    Input : eligible tickers, info DataFrame from filter_ticks
    Output: dict with 'info', 'prices_local' (date x Ticker), 'fx' (USD->CAD by date)"""
    end_plus_one = (pd.Timestamp(end) + pd.Timedelta(days=1)).strftime('%Y-%m-%d')  # yfinance's end date is exclusive

    # one download for all the stocks plus the exchange rate
    # auto_adjust=False gives plain closing prices 
    close = yf.download(list(tickers) + ['USDCAD=X'], start=start, end=end_plus_one,
                        auto_adjust=False, progress=False)['Close']

    fx = close['USDCAD=X'].ffill()                               # exchange rate by date
    prices_local = close.drop(columns='USDCAD=X')
    prices_local = prices_local.dropna(axis=1, how='all')        # drop tickers that failed to download
    prices_local = prices_local.loc[:, prices_local.loc[end].notna()]   # drop tickers with no price on the purchase date
    prices_local = prices_local.ffill()                          # fill holiday gaps with the previous close

    sp500 = yf.download('^GSPC', start=start_date, end=end_date, progress=False)['Close']['^GSPC'] #gets S&P500 close prices in USD
    prices_local['S&P500'] = sp500


    return {"info": info.loc[prices_local.columns], "prices_local": prices_local, "fx": fx}

# 4. CAD conversion, returns, statistics
def to_cad_prices(prices_local, info, fx):
    """Convert USD stock prices to CAD (CAD stocks stay the same).
    Input : prices_local, info, fx    Output: prices_cad (same shape as prices_local)"""
    prices_cad = prices_local.copy()
    for ticker in prices_cad.columns:
        if info.loc[ticker, "currency"] == "USD":
            prices_cad[ticker] = prices_local[ticker] * fx   # price in USD x CAD per 1 USD
    return prices_cad


def compute_returns(prices_cad):
    """Daily returns in CAD: (today's price - yesterday's price) / yesterday's price."""
    return prices_cad.pct_change().dropna(how="all")


def summary_stats(returns):
    """Yearly return and volatility for each stock.
    Output: DataFrame indexed by ticker with columns ann_return and ann_vol."""
    stats = pd.DataFrame()
    stats["ann_return"] = returns.mean() * 252             # average daily return x 252 trading days
    stats["ann_vol"] = returns.std() * (252 ** 0.5)        # daily volatility x square root of 252
    return stats

# 5. Weight caps
def cap_weights(weights, sectors, max_weight, max_sector):
    """Enforce the single-stock and sector caps and keep the weights summing to 1.
    Input : weights Series (index = ticker), sectors Series (same index), the two caps
    Output: Series of adjusted weights"""
    w = (weights / weights.sum()).copy()   # start with weights that sum to 1
    stuck = []                             # stocks already at a limit (they can't take more weight)

    for _ in range(100):                   # repeat until everything fits
        # 1: cut any stock that is above the single-stock cap
        for t in w.index:
            if w[t] > max_weight + 1e-9:
                w[t] = max_weight
                stuck.append(t)

        # 2: cut any sector that is above the sector cap (shrink all its stocks equally)
        for s in sectors.unique():
            names = [t for t in w.index if sectors[t] == s]
            sector_total = sum(w[t] for t in names)
            if sector_total > max_sector + 1e-9:
                for t in names:
                    w[t] = w[t] * max_sector / sector_total
                    stuck.append(t)

        # 3: weight removed by the cuts must go somewhere, so spread it over the other stocks
        leftover = 1 - w.sum()
        if leftover < 1e-9:                # nothing left to give away, so we're done
            break
        free = [t for t in w.index if t not in stuck]
        if len(free) == 0:
            raise ValueError("Caps cannot be met with the selected stocks.")
        free_total = sum(w[t] for t in free)
        for t in free:
            w[t] = w[t] + leftover * w[t] / free_total   # bigger stocks get a bigger share

    return w


# 6. Fees, shares, portfolio table
def transaction_fee_cad(shares, usdcad):
    """Total fees in CAD for buying these shares.
    Each stock's fee is the smaller of US$2.15 and US$0.001 per share, converted to CAD.
    Input : shares Series, USD->CAD rate    Output: total fee (a number)"""
    total_fee = 0
    for number_of_shares in shares:
        fee_usd = min(FEE_CAP_USD, FEE_PER_SHARE_USD * number_of_shares)
        total_fee = total_fee + fee_usd * usdcad
    return total_fee


def allocate_shares(weights, price_cad, usdcad, budget):
    """Turn weights into shares so that purchases + fees = budget.
    The fee depends on the shares and the shares depend on the fee, so we repeat the
    calculation a few times until the numbers settle.
    Output: (shares Series, total_fees number)"""
    fees = 0
    for _ in range(10):
        shares = weights * (budget - fees) / price_cad   # spend what's left after fees
        fees = transaction_fee_cad(shares, usdcad)       # recalculate fees for those shares
    return shares, fees


def build_portfolio(shares, prices_local_buy, price_cad_buy, info):
    """Required output table, indexed 1..n, columns in this order:
    Ticker, Price (local currency), Currency, Shares, Value (CAD), Weight (value / total value)."""
    portfolio = pd.DataFrame(index=shares.index)
    portfolio["Price"] = prices_local_buy[shares.index]
    portfolio["Currency"] = info.loc[shares.index, "currency"]
    portfolio["Shares"] = shares
    portfolio["Value"] = shares * price_cad_buy[shares.index]
    portfolio["Weight"] = portfolio["Value"] / portfolio["Value"].sum()

    portfolio.index.name = "Ticker"
    portfolio = portfolio.reset_index()                   # turn the ticker index into a column
    portfolio.index = range(1, len(portfolio) + 1)        # number rows 1 to n
    return portfolio

def export_csv(portfolio, client):
    """Write Stocks_Group_XX_<client>.csv with only Ticker and Shares."""
    portfolio[["Ticker", "Shares"]].to_csv(f"Stocks_Group_{GROUP_NUMBER}_{client}.csv", index=False)

#filtered, info = filter_ticks(load_tickers())
#print(filtered)
#print(info)


$AGN: possibly delisted; no timezone found
$BK: possibly delisted; no price data found  (1d 2025-10-01 -> 2026-10-01)
$CELG: possibly delisted; no timezone found
$MON: possibly delisted; no timezone found
$RTN: possibly delisted; no timezone found


['AAPL', 'ABBV', 'ABT', 'ACN', 'AIG', 'AMZN', 'AXP', 'BA', 'BAC', 'BB.TO', 'BIIB', 'BLK', 'BMY', 'C', 'CAT', 'CL', 'KO', 'LLY', 'LMT', 'MO', 'MRK', 'PEP', 'PFE', 'RY.TO', 'PM', 'PYPL', 'QCOM', 'SHOP.TO', 'T.TO', 'TD.TO', 'TXN', 'UNH', 'UNP', 'UPS', 'USB']
        currency                  sector     market
AAPL         USD              Technology  us_market
ABBV         USD              Healthcare  us_market
ABT          USD              Healthcare  us_market
ACN          USD              Technology  us_market
AIG          USD      Financial Services  us_market
AMZN         USD       Consumer Cyclical  us_market
AXP          USD      Financial Services  us_market
BA           USD             Industrials  us_market
BAC          USD      Financial Services  us_market
BB.TO        CAD              Technology  ca_market
BIIB         USD              Healthcare  us_market
BLK          USD      Financial Services  us_market
BMY          USD              Healthcare  us_market
C            USD

In [ ]:
# ======================= STRATEGY FUNCTIONS =======================
#minimize can only make things smaller, so making the negative return as small as possible makes the real return as large as possible
def negative_portfolio_return(weights, expected):
    total = 0
    for i in range(len(weights)):
        total = total + weights[i] * expected[i]
    return -total

#it is small when the stocks are calm and don't move together, so minimize looks for the weights that make it smallest
def portfolio_variance(weights, returns):
    portfolio_returns = (returns * weights).sum(axis=1)
    return portfolio_returns.var()

def select_stocks_A(returns, info, n):
    """Pick n stocks for Client A (highest expected return).
        Input : returns DataFrame (eligible tickers), info DataFrame, n
        Output: list of n tickers"""
    #collect each eligible stock's ticker, expected return and beta
    tickers = []
    expected = []
    betas = []

    for ticker in returns.columns:
        tickers.append(ticker)
        expected.append(expected_returns(ticker, returns))
        betas.append(beta(ticker))

    #put the lists into one table with each stock's sector
    stats = pd.DataFrame(index=tickers)
    stats["sector"] = info.loc[tickers, "sector"]
    stats["expected_return"] = expected
    stats["beta"] = betas
    stats = stats.dropna()

    #rank both measures and add the ranks into a score, lower score is a better stock
    stats["rank_return"] = stats["expected_return"].rank(ascending=False)
    stats["rank_beta"] = stats["beta"].rank(ascending=False)
    stats["score"] = stats["rank_return"] + stats["rank_beta"]
    stats = stats.sort_values(["score", "expected_return"], ascending=[True, False])
    stats = stats[stats["expected_return"] > 0]

    #go down the ranking and pick stocks, taking at most 40% * n from the same sector
    max_per_sector = int(CLIENTS["A"]["max_sector"] * n)
    selected = []
    for ticker in stats.index:
        if len(selected) == n:
            break
        sector = stats.loc[ticker, "sector"]
        count = 0
        for chosen in selected:
            if stats.loc[chosen, "sector"] == sector:
                count = count + 1
        if count < max_per_sector:
            selected.append(ticker)

    return selected

def beta(ticker):
    return yf.Ticker(ticker).info.get('beta')

def volatility(ticker, returns):
    return returns[ticker].std()  # this is daily volatitily, to get yearly: * (252 ** 0.5)
    
def expected_returns(ticker, returns):
    return returns[ticker].mean() #expected daily return --> to get monthly: * (21 ** 0.5) --> to get yearly: * (252 ** 0.5)

def weight_stocks_A(selected, returns, info):
    """Raw weights for Client A (before caps). Output: Series indexed by ticker, summing to 1"""
    #limits: 15% per stock and 40% per sector
    max_weight = CLIENTS["A"]["max_weight"]
    max_sector = CLIENTS["A"]["max_sector"]

    n = len(selected)
    expected = list(returns[selected].mean())

    #for each sector make a row of ones (if stock is in this sector) and zeros (if it isn't)
    sector_rows = []
    for sector in info.loc[selected, "sector"].unique():
        row = []
        for ticker in selected:
            if info.loc[ticker, "sector"] == sector:
                row.append(1)
            else:
                row.append(0)
        sector_rows.append(row)

    
    start = [1 / n] * n
    bounds = [(0.01, max_weight)] * n
    constraints = [LinearConstraint([1] * n, 1, 1), LinearConstraint(sector_rows, 0, max_sector)]

    #try different weights within these rules and keep the ones with the highest portfolio expected return
    result = minimize(negative_portfolio_return, start, args=(expected,), method="SLSQP", bounds=bounds, constraints=constraints, tol=1e-12)
    
    return pd.Series(result.x, index=selected)


def select_stocks_B(returns, info, n):
    """Pick n stocks for Client B (lowest volatility). Output: list of n tickers"""
    #collect each eligible stock's ticker, standard deviation and beta
    tickers = []
    std = []
    betas = []

    for ticker in returns.columns:
        tickers.append(ticker)
        std.append(volatility(ticker, returns))
        betas.append(beta(ticker))

    #put the lists into one table with each stock's sector
    stats = pd.DataFrame(index=tickers)
    stats["sector"] = info.loc[tickers, "sector"]
    stats["standard_deviation"] = std
    stats["beta"] = betas
    stats = stats.dropna()

    #rank both measures and add the ranks into a score, lower score is a better stock
    stats["rank_volatility"] = stats["standard_deviation"].rank(ascending=True)
    stats["rank_beta"] = stats["beta"].rank(ascending=True)
    stats["score"] = stats["rank_volatility"] + stats["rank_beta"]
    stats = stats.sort_values(["score", "standard_deviation"], ascending=[True, True])
    stats = stats[stats["standard_deviation"] > 0]

    #go down the ranking and pick stocks, taking at most 40% * n from the same sector
    max_per_sector = int(CLIENTS["B"]["max_sector"] * n)
    selected = []
    for ticker in stats.index:
        if len(selected) == n:
            break
        sector = stats.loc[ticker, "sector"]
        count = 0
        for chosen in selected:
            if stats.loc[chosen, "sector"] == sector:
                count = count + 1
        if count < max_per_sector:
            selected.append(ticker)

    return selected


def weight_stocks_B(selected, returns, info):
    """Raw weights for Client B (before caps). Output: Series indexed by ticker, summing to 1"""
    #limits: 15% per stock and 40% per sector
    max_weight = CLIENTS["B"]["max_weight"]
    max_sector = CLIENTS["B"]["max_sector"]
    n = len(selected)

    #daily returns of the picked stocks, only the days on which all of them have data
    stock_returns = returns[selected].dropna()

    #for each sector make a row of ones (if stock is in this sector) and zeros (if it isn't)
    sector_rows = []
    for sector in info.loc[selected, "sector"].unique():
        row = []
        for ticker in selected:
            if info.loc[ticker, "sector"] == sector:
                row.append(1)
            else:
                row.append(0)
        sector_rows.append(row)

    start = [1 / n] * n
    bounds = [(0.01, max_weight)] * n
    constraints = [LinearConstraint([1] * n, 1, 1), LinearConstraint(sector_rows, 0, max_sector)]

    #try different weights within these rules and keep the ones with the lowest portfolio variance
    result = minimize(portfolio_variance, start, args=(stock_returns,), method="SLSQP", bounds=bounds, constraints=constraints, tol=1e-12)
    return pd.Series(result.x, index=selected)


In [ ]:
def finish_portfolio(client, selected, raw_weights, data, returns, prices_cad, usdcad):
    """Everything after stock selection and weighting, the same for both clients:
    apply caps, buy shares, build the table.
    Output: (portfolio table, final weights, total fees)"""
    cfg = CLIENTS[client]
    info = data["info"]

    weights = cap_weights(raw_weights, info.loc[selected, "sector"], cfg["max_weight"], cfg["max_sector"])

    price_cad_buy = prices_cad.loc[PURCHASE_DATE]
    shares, fees = allocate_shares(weights, price_cad_buy[selected], usdcad, cfg["budget"])
    portfolio = build_portfolio(shares, data["prices_local"].loc[PURCHASE_DATE], price_cad_buy, info)

    return portfolio, weights, fees


# load and prepare the data
tickers = load_tickers(TICKER_FILE)
eligible, info = filter_ticks(tickers)
data = load_market_data(eligible, info)
info = data["info"]
eligible = list(info.index)            # after dropping tickers with no purchase-date price

prices_cad = to_cad_prices(data["prices_local"], info, data["fx"])
returns = compute_returns(prices_cad)
usdcad = data["fx"].loc[PURCHASE_DATE]

## Client A — Growth

Construct a portfolio intended to achieve the **highest expected return**. Use your selected stocks and portfolio weights to create a DataFrame named `Portfolio_Final_A` with index **1 through n** and columns (in this order): `Ticker`, `Price`, `Currency`, `Shares`, `Value`, `Weight`. `Price` is in the stock’s quoted currency, `Value` is in CAD, and `Weight` is a fraction summing to 1.

In [ ]:
N_STOCKS_A = 10    # TODO: agree on the count (10-20)

selected_A = select_stocks_A(returns[eligible], info, N_STOCKS_A)
raw_weights_A = weight_stocks_A(selected_A, returns, info)
Portfolio_Final_A, weights_A, fees_A = finish_portfolio("A", selected_A, raw_weights_A, data, returns, prices_cad, usdcad)



### Client A — Financial justification

**Why does this portfolio make financial sense?** Explain how and why you select and weight stocks.

**What evidence supports your strategy?** Discuss relevant statistics and the equal-weight comparison below.

**What could go wrong?** Identify at least one important limitation or situation in which this strategy may perform poorly.

## Client B — Low Risk

Construct a portfolio intended to achieve the **lowest risk**, defined as the standard deviation of daily portfolio returns in CAD. Create `Portfolio_Final_B` using the same DataFrame column names, order, and index convention as for Client A.

In [ ]:
N_STOCKS_B = 15    # TODO: agree on the count (10-20)

selected_B = select_stocks_B(returns[eligible], info, N_STOCKS_B)
raw_weights_B = weight_stocks_B(selected_B, returns, info)
Portfolio_Final_B, weights_B, fees_B = finish_portfolio("B", selected_B, raw_weights_B, data, returns, prices_cad, usdcad)

### Client B — Financial justification

**Why does this portfolio make financial sense?** Explain how and why you select and weight stocks.

**What evidence supports your strategy?** Discuss relevant statistics and the equal-weight comparison below.

**What could go wrong?** Identify at least one important limitation or situation in which this strategy may perform poorly.

## Equal-weight comparisons

For **each client**, create an equal-weighted comparison portfolio holding **exactly the same stocks** as that client’s chosen portfolio, with starting weight `1/n` per stock. Over the same clearly identified historical period, calculate **historical return** and **volatility** for the chosen portfolio and its equal-weighted counterpart. Use consistent CAD valuations and weighting assumptions.

Display one concise comparison table per client, and interpret the results in the financial justifications above. No additional graph or out-of-sample test is required.

In [ ]:
def portfolio_stats(weights, returns):
    """Historical yearly return and volatility of a portfolio with fixed weights.
    Output: dict with 'Return' and 'Volatility'"""
    stock_returns = returns.loc[HIST_START:HIST_END, weights.index].dropna()
    portfolio_returns = (stock_returns * weights).sum(axis=1)   # each day: sum of weight x stock return
    return {"Return": portfolio_returns.mean() * 252,
            "Volatility": portfolio_returns.std() * (252 ** 0.5)}


def equal_weight_comparison(final_weights, returns):
    """Compare your strategy to an equal-weight portfolio of the same stocks.
    Output: 2-row DataFrame with columns Return and Volatility (in % per year)."""
    n = len(final_weights)
    equal_weights = pd.Series(1 / n, index=final_weights.index)   # every stock gets 1/n

    strategy = portfolio_stats(final_weights, returns)
    equal = portfolio_stats(equal_weights, returns)
    table = pd.DataFrame([strategy, equal], index=["Strategy", "Equal-weight"])
    return (table * 100).round(2)


print(f"Historical period: {HIST_START} to {HIST_END}. Daily returns in CAD, weights held constant "
      f"(rebalanced daily), annualized with 252 trading days. Values are in % per year.")

print("\nClient A (Growth)")
display(equal_weight_comparison(weights_A, returns))

print("\nClient B (Low Risk)")
display(equal_weight_comparison(weights_B, returns))

## Programmatic validation

Use Python (`assert` statements or a simple function are fine) to display whether **both** final portfolios pass each of the following checks:

- Selected tickers are valid and eligible.
- Number of stocks is between 10 and 20.
- No individual stock weight exceeds 15%.
- No single sector has more than 40% of portfolio value.
- Portfolio weights sum to 100%.
- Total stock purchases plus transaction fees do not exceed the applicable client budget.

Your code should run with the TAs’ ticker file without manually changing individual stock selections.

In [ ]:
def validate_portfolio(portfolio, info, eligible, cfg, fees):
    """Check every required rule and print PASS or FAIL for each."""
    # add up the weight in each sector
    sector_weights = {}
    for i in portfolio.index:
        sector = info.loc[portfolio.loc[i, "Ticker"], "sector"]
        sector_weights[sector] = sector_weights.get(sector, 0) + portfolio.loc[i, "Weight"]

    total_spent = portfolio["Value"].sum() + fees

    tests = [
        ("Tickers valid and eligible", all(t in eligible for t in portfolio["Ticker"])),
        ("Stock count between 10 and 20", cfg["n_min"] <= len(portfolio) <= cfg["n_max"]),
        ("No stock above max weight",  portfolio["Weight"].max() <= cfg["max_weight"] + 1e-6),
        ("No sector above max weight", max(sector_weights.values()) <= cfg["max_sector"] + 1e-6),
        ("Weights sum to 100%",        abs(portfolio["Weight"].sum() - 1) < 1e-6),
        ("Purchases + fees within budget", total_spent <= cfg["budget"] + 1e-6),
    ]
    for name, passed in tests:
        print(f"{name}: {'PASS' if passed else 'FAIL'}")


print("Client A (Growth)")
validate_portfolio(Portfolio_Final_A, info, eligible, CLIENTS["A"], fees_A)

print("\nClient B (Low Risk)")
validate_portfolio(Portfolio_Final_B, info, eligible, CLIENTS["B"], fees_B)

## Final portfolio outputs

Display `Portfolio_Final_A` and `Portfolio_Final_B`, along with the budget reconciliation (purchases plus fees) for each. Generate `Stocks_Group_XX_A.csv` and `Stocks_Group_XX_B.csv` (replace `XX` with your team number). Each CSV must contain **only** the columns `Ticker` and `Shares`. The notebook must **generate** these CSV files when run, but you submit **only the notebook** to Dropbox.

In [ ]:
def show_final_output(client, portfolio, fees):
    """Display the portfolio, show the budget reconciliation, and save the CSV."""
    cfg = CLIENTS[client]
    purchases = portfolio["Value"].sum()

    print(f"===== Portfolio_Final_{client}: {cfg['name']} =====")
    display(portfolio)

    print("Budget reconciliation (CAD):")
    print(f"  Stock purchases:  {purchases:,.2f}")
    print(f"  Transaction fees: {fees:,.2f}")
    print(f"  Total spent:      {purchases + fees:,.2f}")
    print(f"  Client budget:    {cfg['budget']:,.2f}")
    print(f"  Unspent:          {cfg['budget'] - purchases - fees:,.2f}")

    export_csv(portfolio, client)       # writes Ticker and Shares only
    print(f"  Saved: Stocks_Group_{GROUP_NUMBER}_{client}.csv\n")


show_final_output("A", Portfolio_Final_A, fees_A)
show_final_output("B", Portfolio_Final_B, fees_B)

## Contribution and AI Declaration

**The following team members made a meaningful contribution to this assignment:**

Enter the names of all contributing members below (one name per line):

**To the best of our knowledge, no generative AI was used in preparing this submission.**

List only members who made a meaningful contribution. Contact the instructor before the deadline if someone should be omitted or if the team cannot make the declaration.